# Script 3: Ecosystem Audit

### Context
Notebooks 1 and 2 proved the problem formally. This notebook provides real-world evidence that the consequences are visible and measurable in the ecosystem.

We audit Hugging Face in two ways:
1. Check what safety documentation exists on the official model
2. Find applications built on Stable Diffusion that advertise NSFW or uncensored content

We then use Z3 to formally state: if an application built on this model is NSFW, the licence constraint is violated.

### Step 1: Imports

In [1]:
import requests
import json
from z3 import *

### Step 2: Check Official Model Safety Documentation

We first confirm what Stability AI has publicly documented about safety on the model page itself.

In [2]:
url = "https://huggingface.co/api/models/stabilityai/stable-diffusion-xl-base-1.0"
metadata = requests.get(url, timeout=10).json()

tags = metadata.get('tags', [])
downloads = metadata.get('downloads', 0)
likes = metadata.get('likes', 0)
card_data = metadata.get('cardData', {}) or {}

safety_tags = [t for t in tags if any(
    term in t.lower() for term in ['safety', 'nsfw', 'content', 'filter']
)]

print(f"Model:      stabilityai/stable-diffusion-xl-base-1.0")
print(f"Downloads:  {downloads:,}")
print(f"Likes:      {likes:,}")
print(f"Licence:    {card_data.get('license', 'Not specified')}")
print(f"Safety tags: {safety_tags if safety_tags else 'None found'}")

Model:      stabilityai/stable-diffusion-xl-base-1.0
Downloads:  2,091,235
Likes:      7,556
Licence:    openrail++
Safety tags: None found


**No safety-related tags exist on the model page**. Safety is addressed only through the licence text, with no enforcement mechanism documented.


### Step 3: Search for NSFW Applications

We search across multiple Stable Diffusion model families to find applications advertising removed safety mechanisms. We check SDXL 1.0, SD 1.5, and SD 2.1 since derivative applications are spread across all versions.

In [3]:
nsfw_terms = ['nsfw', 'uncensored', 'adult', 'explicit',
              'lustly', 'booby', 'xxx', 'nude', 'erotic']

models_to_check = [
    "stabilityai/stable-diffusion-xl-base-1.0",
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "stabilityai/stable-diffusion-2-1",
]

all_nsfw_spaces = []
all_spaces = []

for model_id in models_to_check:
    response = requests.get(
        "https://huggingface.co/api/spaces",
        params={"model": model_id, "limit": 100},
        timeout=15
    )
    spaces = response.json()
    all_spaces.extend(spaces)

    nsfw_found = []
    for space in spaces:
        space_id = space.get('id', '').lower()
        tags_str = ' '.join(space.get('tags', [])).lower()
        if any(term in space_id or term in tags_str for term in nsfw_terms):
            nsfw_found.append({
                'model': model_id,
                'id': space.get('id'),
                'likes': space.get('likes', 0)
            })

    print(f"Model: {model_id}")
    print(f"  Total spaces: {len(spaces)} | NSFW found: {len(nsfw_found)}")
    all_nsfw_spaces.extend(nsfw_found)

print(f"\nTotal spaces audited: {len(all_spaces)}")
print(f"Total NSFW spaces:    {len(all_nsfw_spaces)}")

Model: stabilityai/stable-diffusion-xl-base-1.0
  Total spaces: 100 | NSFW found: 7
Model: stable-diffusion-v1-5/stable-diffusion-v1-5
  Total spaces: 100 | NSFW found: 7
Model: stabilityai/stable-diffusion-2-1
  Total spaces: 100 | NSFW found: 7

Total spaces audited: 300
Total NSFW spaces:    21


### Step 4: NSFW Applications Found

In [4]:
print("NSFW or uncensored applications found:\n")
for s in all_nsfw_spaces:
    print(f"  {s['id']}")
    print(f"    Base model: {s['model']} | Likes: {s['likes']}")

NSFW or uncensored applications found:

  Heartsync/Adult
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 616
  Heartsync/NSFW-Uncensored-photo
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 749
  BoobyBoobs/Flux_Lustly_AI_Uncensored_NSFW_V1
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 85
  Imosu/image_audio_to_video_NSFW
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 16
  Landol4/nsfw-face-swap
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 12
  Menyu/wainsfw
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 210
  rikunarita/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-GGUF-Q6_K
    Base model: stabilityai/stable-diffusion-xl-base-1.0 | Likes: 10
  Heartsync/Adult
    Base model: stable-diffusion-v1-5/stable-diffusion-v1-5 | Likes: 616
  Heartsync/NSFW-Uncensored-photo
    Base model: stable-diffusion-v1-5/stable-diffusion-v1-5 | Likes: 749
  BoobyBoobs/Flux_Lustly_AI_Uncensore

### Step 5: Formal Constraint Check with Z3

The licence for Stable Diffusion states the model must not be used to generate harmful content. We encode this as a Z3 constraint and check whether any of the NSFW applications found above violate it.

If an application is built on the model AND is NSFW, then the licence constraint is violated. Z3 checks whether this situation can exist without a violation occurring.

In [ ]:
App = DeclareSort('App')
BuiltOnSD = Function('BuiltOnSD', App, BoolSort())
IsNSFW = Function('IsNSFW',    App, BoolSort())
Violates = Function('Violates',  App, BoolSort())

app = Const('app', App)
s = Solver()

s.add(ForAll([app],
    Implies(
        And(BuiltOnSD(app), IsNSFW(app)),
        Violates(app)
    )
))

example_app = Const('nsfw_app', App)
s.add(BuiltOnSD(example_app))
s.add(IsNSFW(example_app))
s.add(Not(Violates(example_app)))

result = s.check()

print("Z3 Licence Constraint Check")
print("="*50)
print("Rule:   BuiltOnSD AND IsNSFW -> Violates licence")
print("Fact:   An NSFW application built on SD exists")
print("Query:  Can it exist WITHOUT violating the licence?")
print()
print(f"Z3 Result: {result}")
if result == unsat:
    print("UNSAT: Every NSFW application built on Stable Diffusion")
    print("necessarily violates the licence constraint.")
    print("The existence of the applications found above is a")
    print("formal licence violation under the model's own terms.")

Z3 Licence Constraint Check
Rule:   BuiltOnSD AND IsNSFW -> Violates licence
Fact:   An NSFW application built on SD exists
Query:  Can it exist WITHOUT violating the licence?

Z3 Result: unsat
UNSAT: Every NSFW application built on Stable Diffusion
necessarily violates the licence constraint.
The existence of the applications found above is a
formal licence violation under the model's own terms.


### Step 6: Save Results

In [6]:
output = {
    "total_spaces_audited": len(all_spaces),
    "nsfw_spaces_found": len(all_nsfw_spaces),
    "nsfw_spaces": all_nsfw_spaces
}

with open("audit_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved. {len(all_nsfw_spaces)} NSFW spaces documented.")

Results saved. 21 NSFW spaces documented.


### Summary

| Check | Result |
|---|---|
| Safety tags on official model | None found |
| NSFW applications across SD model family | Found across all versions |
| Z3 licence constraint | UNSAT: violation is formally proven |

The audit confirms that the design decision examined in Notebooks 1 and 2 has real measurable consequences. Applications built directly 
on Stable Diffusion and advertising NSFW or uncensored content are publicly accessible on the same platform that distributes the model.

Z3 formally proves that every such application necessarily violates the model's own licence terms. The ethical responsibility Stability AI 
stated in their licence has not been enforced in practice.